If using Mac M2 -> change gpu=0 to gpu='mps' 
Also comment out google drive import and uncomment BASE_STORAGE
Also will have to zip data in 50 seeds batches using terminal

In [1]:
!nvidia-smi

Sun Jun 21 06:12:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive  
drive.mount('/content/drive')

# FOR MAC M2 ONLY
#BASE_STORAGE = "./NeuralMAG_Data"

Mounted at /content/drive


In [11]:
# 2. Change to the ROOT folder (not the utils folder)
os.chdir('/content/drive/MyDrive/NeuralMAG_Data') 

In [23]:
import os
import numpy as np
import random
from utils.gen_data import prepare_model, simulate_spins, save_simulation_data
from libs.misc import initial_spin_prepare, create_random_mask

In [24]:
# 1. Global Setup
GRID_W = 96
TOTAL_SEEDS = 300       # To get 150k samples (140k train / 10k val)
MASK_THRESHOLD = 200    # First 200 seeds (2/3) get random masks 
BASE_SAVE_DIR = "/content/drive/MyDrive/NeuralMAG_Data/size32"

In [25]:
# 2. Argument Class (Simulates Command Line Args)
class Args:
    w = GRID_W
    layers = 2          # Bilayer thin film 
    Ms = 1000           # Default Saturation Magnetization 
    Ax = 0.5e-6         # Default Exchange constant 
    Ku = 0.0            # No anisotropy for training set 
    Kvec = (1.0, 0, 0)
    gpu = 0             # Use 0 for Colab; update to 'mps' logic if on M2
    sav_samples = 500   # To match original paper sample count 

    # need to define all below to avoid error like:  'Args' object has no attribute 'error_min'
    dtime = 1.0e-13       # CRITICAL: Physical time step in seconds 
    error_min = 1.0e-5    # Convergence threshold (error must drop below this) 
    max_iter = 50000      # Safety limit to prevent infinite loops 
    damping = 0.1           # Damping coefficient
    cell = (3, 3, 3)      # Cell size in nm

args = Args()

In [26]:
# Generate all seeds
for seed_id in range(TOTAL_SEEDS):
    is_masked = seed_id < MASK_THRESHOLD
    mask_label = "masked" if is_masked else "unmasked"
    
    # 1. Randomize External Field (100-1000 Oe) and orientation in-plane 
    Hext_val = np.random.uniform(100, 1000)
    theta = np.random.uniform(0, 2 * np.pi)
    Hext_vec = np.array([np.cos(theta), np.sin(theta), 0.0])
    Hext = Hext_val * Hext_vec

    # 2. Folder Naming: includes seed number, grid size, mask status, and Hext value
    seed_folder = f"seed_{seed_id:03d}_size{args.w}_{mask_label}_Hext_{int(Hext_val)}"
    seed_path = os.path.join(BASE_SAVE_DIR, f"size{args.w}_general", seed_folder)
    os.makedirs(seed_path, exist_ok=True)

    # 3. Model and Spin Initialization 
    film = prepare_model(args)
    # Randomized magnetization within the film plane (X-Y) 
    spin = initial_spin_prepare(args.w, args.layers, seed_id) 

    if is_masked:
        # Random polygon generation using authors' randomization parameters
        mask = create_random_mask((args.w, args.w), np.random.randint(2, args.w), random.choice([True, False]))
        # Physically set magnetization to zero in the masked area 
        spin = film.SpinInit(spin * mask)
    else:
        spin = film.SpinInit(spin)

    print(f"--- Seed {seed_id:03d} | Hext: {Hext_val:.1f} Oe | {mask_label} ---")
    
    # 4. Run FFT/LLG simulation and return trajectory lists 
    Spins_list, Hds_list, error_list = simulate_spins(film, spin, Hext, args)

    # 5. Save all three data types per seed (Calls modified save_simulation_data)
    save_simulation_data(Spins_list, Hds_list, args, seed_path, error_list, Hext)

print(f"Generation Complete for Size {args.w}x{args.w}.")


Initialize an mmModel:

  Cuda:0 available. Spin evolution using cuda:0.

  # Cells  : 18432
  # Matters: 1
  Ms check : 1st 1000.000, last 1000.000   [emu/cc]
  Ax check : 1st 5.00e-07, last 5.00e-07   [erg/cm]
  Ku check : 1st 0.00e+00, last 0.00e+00   [erg/cc]

  Average magnetization  :   1000.00  [emu/cc]
  Maximal anisotropy Hk  : 0.000e+00  [Oe]    
  Maximal Heisenberg Hx  : 1.111e+04  [Oe]    
  Maximal effective Heff : 7.923e+04  [Oe]
  
Creating 2 layer models 

Demag Matrix SumTest:
  [ 2.63260e-01 -1.69342e-01 -9.16640e-02]
  [-1.69342e-01  2.63260e-01 -9.16640e-02]
  [-9.16640e-02 -9.16640e-02  4.73481e-01]
  trace = 1.00000

Time cost: 2.351269 s for initializing demag matrix 

Spin state initialized according to input.

--- Seed 000 | Hext: 905.3 Oe | masked ---

Initialize an mmModel:

  Cuda:0 available. Spin evolution using cuda:0.

  # Cells  : 18432
  # Matters: 1
  Ms check : 1st 1000.000, last 1000.000   [emu/cc]
  Ax check : 1st 5.00e-07, last 5.00e-07   [erg/c